# LLM Labeling for Ground Truth

This notebook labels the shuffled candidate pool produced by `1_build_groundtruth.ipynb`.

The labeling strategy is query-batched:

1. Load `blinded_annotation_items.jsonl`.
2. Group candidates by `query_id`.
3. Send one query and all of its candidate documents to the LLM.
4. Parse the returned JSON labels.
5. Save flattened pair-level labels to `llm_groundtruth_labels.jsonl`.

With 500 queries, this means about 500 LLM calls rather than one call per query-document pair.

## 1. Imports and Environment Loading

In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Optional

import pandas as pd

In [ ]:
def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


def find_repo_root(finalproject_root: Path) -> Path:
    """Return the repository root that contains Finalproject."""
    return finalproject_root.parent


def load_env_file(env_path: Path) -> None:
    """Load key=value pairs from a .env file without overriding existing environment variables."""
    if not env_path.exists():
        print(f"No .env file found at: {env_path}")
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


def get_env_int(name: str, default: int) -> int:
    value = os.getenv(name)
    return default if value in (None, "") else int(value)


def get_env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    return default if value in (None, "") else float(value)


FINALPROJECT_ROOT = find_finalproject_root()
REPO_ROOT = find_repo_root(FINALPROJECT_ROOT)
ENV_PATH = REPO_ROOT / ".env"
load_env_file(ENV_PATH)

DATA_PATH = FINALPROJECT_ROOT / "data" / "all_recipes_final.csv"
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
ANNOTATION_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "annotation"
BLINDED_ANNOTATION_ITEMS_PATH = ANNOTATION_OUTPUT_DIR / "blinded_annotation_items.jsonl"

print("Finalproject root:", FINALPROJECT_ROOT)
print(".env path:", ENV_PATH)
print("Blinded annotation items:", BLINDED_ANNOTATION_ITEMS_PATH)

## 2. Main Configuration

In [ ]:
# -------------------------------------------------------
# OpenAI-compatible LiteLLM labeling config
# -------------------------------------------------------
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://litellm.imt-soft/v1")
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME", "gpt-4o-mini")
LLM_API_KEY = os.getenv("LLM_API_KEY", "not-needed")

# Self-hosted endpoint defaults: generous response budget and timeout.
LLM_TEMPERATURE = get_env_float("LLM_TEMPERATURE", 0.0)
LLM_MAX_TOKENS = get_env_int("LLM_MAX_TOKENS", 16384)
LLM_REQUEST_TIMEOUT_SECONDS = get_env_int("LLM_REQUEST_TIMEOUT_SECONDS", 300)
LLM_MAX_RETRIES = get_env_int("LLM_MAX_RETRIES", 5)

# None means label all query groups. Set a small integer for debugging.
MAX_QUERY_GROUPS_TO_LABEL = None

RESUME_LLM_LABELING = True
LLM_LABELS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_labels.jsonl"
LLM_QUERY_LOGS_PATH = ANNOTATION_OUTPUT_DIR / "llm_groundtruth_query_logs.jsonl"


def normalize_openai_compatible_base_url(raw_base_url: str) -> str:
    """Normalize LiteLLM proxy URL for OpenAI-compatible SDK clients."""
    base_url = str(raw_base_url).strip().rstrip("/")
    if not base_url:
        raise ValueError("LLM_BASE_URL cannot be empty.")
    if not base_url.startswith(("http://", "https://")):
        base_url = "https://" + base_url
    if not base_url.endswith("/v1"):
        base_url = base_url + "/v1"
    return base_url


LLM_BASE_URL = normalize_openai_compatible_base_url(LLM_BASE_URL)

print("LLM base URL:", LLM_BASE_URL)
print("LLM model:", LLM_MODEL_NAME)
print("Pair-level labels output:", LLM_LABELS_PATH)
print("Query-level logs output:", LLM_QUERY_LOGS_PATH)

## 3. Load Shuffled Annotation Items

In [ ]:
def load_jsonl_records(jsonl_path: Path) -> list[dict]:
    """Load a JSONL file into a list of dictionaries."""
    records = []
    with jsonl_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if line.strip():
                records.append(json.loads(line))
    return records


def append_jsonl_record(output_path: Path, record: dict) -> None:
    """Append one JSON-serializable record to a JSONL file."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("a", encoding="utf-8") as output_file:
        output_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        output_file.flush()


def compact_text(value: str, max_characters: int) -> str:
    """Clip long text fields so one query prompt remains manageable."""
    text = str(value or "").strip()
    if len(text) > max_characters:
        return text[: max_characters - 3].rstrip() + "..."
    return text


def group_annotation_items_by_query(annotation_items: list[dict]) -> list[dict]:
    """Group shuffled annotation items into one prompt payload per query."""
    grouped: dict[int, dict] = {}
    for item in annotation_items:
        query_id = int(item["query_id"])
        grouped.setdefault(
            query_id,
            {
                "query_id": query_id,
                "query_text": str(item["query_text"]),
                "documents": [],
            },
        )
        grouped[query_id]["documents"].append(item)

    query_groups = []
    for query_group in grouped.values():
        query_group["documents"] = sorted(
            query_group["documents"],
            key=lambda item: int(item["blinded_position"]),
        )
        query_groups.append(query_group)
    return sorted(query_groups, key=lambda group: group["query_id"])


def build_document_payload(annotation_item: dict) -> dict:
    """Build the compact document object sent to the LLM."""
    return {
        "blinded_position": int(annotation_item["blinded_position"]),
        "doc_id": int(annotation_item["doc_id"]),
        "title": compact_text(annotation_item.get("recipe_title", ""), max_characters=180),
        "recipe_type": compact_text(annotation_item.get("recipe_type", ""), max_characters=80),
        "ingredients": compact_text(annotation_item.get("ingredients", ""), max_characters=900),
        "normalized_ingredients": compact_text(annotation_item.get("normalized_ingredients", ""), max_characters=700),
        "cooking_steps": compact_text(annotation_item.get("cooking_steps", ""), max_characters=1200),
    }


def build_query_payload(query_group: dict) -> dict:
    """Build the full query-level labeling payload."""
    return {
        "query_id": int(query_group["query_id"]),
        "query_text": str(query_group["query_text"]),
        "documents": [build_document_payload(item) for item in query_group["documents"]],
    }


def strip_markdown_json_fence(raw_text: str) -> str:
    """Remove common markdown code fences around JSON output."""
    text = str(raw_text).strip()
    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    return fenced_match.group(1).strip() if fenced_match else text


def parse_query_label_response(raw_response_text: str, expected_payload: dict) -> list[dict]:
    """Parse and validate the LLM JSON labels for one query."""
    cleaned_response = strip_markdown_json_fence(raw_response_text)
    parsed_response = json.loads(cleaned_response)

    if not isinstance(parsed_response, dict):
        raise ValueError("LLM response must be a JSON object.")
    if int(parsed_response.get("query_id")) != int(expected_payload["query_id"]):
        raise ValueError(
            f"LLM response query_id {parsed_response.get('query_id')} does not match expected {expected_payload['query_id']}."
        )

    labels = parsed_response.get("labels")
    if not isinstance(labels, list):
        raise ValueError("LLM response must contain a labels list.")

    expected_keys = {
        (int(document["doc_id"]), int(document["blinded_position"]))
        for document in expected_payload["documents"]
    }
    parsed_records = []
    seen_keys = set()
    for label_item in labels:
        doc_id = int(label_item["doc_id"])
        blinded_position = int(label_item["blinded_position"])
        relevance = int(label_item["relevance"])
        if relevance not in {0, 1, 2, 3}:
            raise ValueError(f"Invalid relevance label {relevance} for doc_id={doc_id}.")
        key = (doc_id, blinded_position)
        if key not in expected_keys:
            raise ValueError(f"Unexpected label key from LLM: {key}")
        if key in seen_keys:
            raise ValueError(f"Duplicate label key from LLM: {key}")
        seen_keys.add(key)
        parsed_records.append(
            {
                "query_id": int(expected_payload["query_id"]),
                "doc_id": doc_id,
                "blinded_position": blinded_position,
                "relevance": relevance,
            }
        )

    missing_keys = expected_keys - seen_keys
    if missing_keys:
        raise ValueError(f"LLM response missed {len(missing_keys)} documents: {sorted(missing_keys)[:5]}")

    return sorted(parsed_records, key=lambda record: record["blinded_position"])

In [ ]:
blinded_annotation_items = load_jsonl_records(BLINDED_ANNOTATION_ITEMS_PATH)
query_groups = group_annotation_items_by_query(blinded_annotation_items)

print("Number of blinded annotation items:", len(blinded_annotation_items))
print("Number of query groups:", len(query_groups))
print("Candidates per query:")
print(pd.Series([len(group["documents"]) for group in query_groups]).describe())

query_groups[0]["query_id"], query_groups[0]["query_text"], len(query_groups[0]["documents"])

## 4. Query-Batched Relevance Prompt

In [ ]:
LLM_JUDGE_PROMPT_VERSION = "recipe_relevance_query_batch_v1"

RELEVANCE_JUDGE_SYSTEM_PROMPT = """You are an expert relevance assessor for a Vietnamese recipe search system.
Your task is to label how relevant each candidate recipe is to the user query.
Use only the provided query and recipe fields.
Return valid JSON only. Do not include markdown, comments, or explanations.
"""

RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE = """TASK
Label every candidate document for the given query using a 0-3 graded relevance scale.

QUERY
query_id: {query_id}
query_text: {query_text}

CANDIDATE DOCUMENTS
{documents_json}

RELEVANCE DEFINITIONS
3 = Highly relevant:
- The recipe directly matches the main dish, main food topic, or main search intent.
- Close variants are acceptable when the main intent is still clearly satisfied.

2 = Relevant:
- The recipe is not the exact dish, but it is a close variant or reasonable substitute.
- It strongly connects to the main intent by dish group, cooking method, or main ingredient.

1 = Somewhat relevant:
- The recipe shares only a weak but meaningful aspect such as a main ingredient, occasion, broad dish type, or general food property.

0 = Not relevant:
- The recipe does not substantially help the query.
- Generic word overlap is not enough.

MANDATORY RULES
- Label every candidate document exactly once.
- Preserve each document's doc_id and blinded_position exactly.
- Prioritize the main search intent over keyword overlap.
- Do not infer facts that are missing from the recipe fields.
- Return JSON only, exactly in this shape:
{{
  "query_id": {query_id},
  "labels": [
    {{"doc_id": 123, "blinded_position": 1, "relevance": 0}},
    {{"doc_id": 456, "blinded_position": 2, "relevance": 3}}
  ]
}}
"""


def build_relevance_judge_prompt_for_query(query_group: dict) -> dict:
    """Build one prompt for one query and all its candidate documents."""
    payload = build_query_payload(query_group)
    documents_json = json.dumps(payload["documents"], ensure_ascii=False, indent=2)
    user_prompt = RELEVANCE_JUDGE_USER_PROMPT_TEMPLATE.format(
        query_id=payload["query_id"],
        query_text=payload["query_text"],
        documents_json=documents_json,
    )
    return {
        "system_prompt": RELEVANCE_JUDGE_SYSTEM_PROMPT,
        "user_prompt": user_prompt,
        "prompt_version": LLM_JUDGE_PROMPT_VERSION,
        "payload": payload,
    }

## 5. OpenAI-Compatible LiteLLM Caller

In [ ]:
def call_openai_compatible_query_judge(system_prompt: str, user_prompt: str) -> str:
    """Call the LiteLLM OpenAI-compatible endpoint and return raw text for one query group."""
    try:
        from openai import OpenAI
    except ImportError as import_error:
        raise ImportError("OpenAI SDK is not installed. Install it with `pip install openai`.") from import_error

    client = OpenAI(
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        timeout=LLM_REQUEST_TIMEOUT_SECONDS,
    )
    response = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
    )
    return response.choices[0].message.content.strip()

## 6. Run Labeling

In [ ]:
def load_completed_query_ids(labels_path: Path) -> set[int]:
    """Return query IDs that already have at least one saved label record."""
    if not labels_path.exists():
        return set()
    completed_records = load_jsonl_records(labels_path)
    return {int(record["query_id"]) for record in completed_records}


def judge_query_groups_with_llm(
    query_groups: list[dict],
    labels_output_path: Path,
    query_logs_output_path: Path,
    resume: bool,
    max_query_groups: Optional[int] = None,
) -> pd.DataFrame:
    """Label query groups with one LLM call per query and save flattened pair-level labels."""
    selected_query_groups = query_groups[:max_query_groups] if max_query_groups is not None else query_groups
    completed_query_ids = load_completed_query_ids(labels_output_path) if resume else set()
    saved_label_records = []

    print(f"Query groups selected: {len(selected_query_groups)}")
    print(f"Already completed query IDs: {len(completed_query_ids)}")

    for index, query_group in enumerate(selected_query_groups, start=1):
        query_id = int(query_group["query_id"])
        if query_id in completed_query_ids:
            continue

        prompt = build_relevance_judge_prompt_for_query(query_group)
        raw_response_text = call_openai_compatible_query_judge(prompt["system_prompt"], prompt["user_prompt"])
        parsed_label_records = parse_query_label_response(raw_response_text, prompt["payload"])

        query_log_record = {
            "query_id": query_id,
            "judge_type": "llm",
            "llm_provider": "openai_compatible_litellm",
            "llm_model_name": LLM_MODEL_NAME,
            "prompt_version": prompt["prompt_version"],
            "candidate_count": len(prompt["payload"]["documents"]),
            "parsed_label_count": len(parsed_label_records),
            "raw_response_text": raw_response_text,
        }
        append_jsonl_record(query_logs_output_path, query_log_record)

        for label_record in parsed_label_records:
            output_record = {
                **label_record,
                "judge_type": "llm",
                "llm_provider": "openai_compatible_litellm",
                "llm_model_name": LLM_MODEL_NAME,
                "prompt_version": prompt["prompt_version"],
            }
            append_jsonl_record(labels_output_path, output_record)
            saved_label_records.append(output_record)

        completed_query_ids.add(query_id)
        print(
            f"[{index}/{len(selected_query_groups)}] query_id={query_id} "
            f"saved {len(parsed_label_records)} labels"
        )

    return pd.DataFrame(saved_label_records)

In [ ]:
# This is the main labeling run.
# It performs one OpenAI-compatible LiteLLM call per query group.
llm_label_dataframe = judge_query_groups_with_llm(
    query_groups=query_groups,
    labels_output_path=LLM_LABELS_PATH,
    query_logs_output_path=LLM_QUERY_LOGS_PATH,
    resume=RESUME_LLM_LABELING,
    max_query_groups=MAX_QUERY_GROUPS_TO_LABEL,
)
llm_label_dataframe.head()

## 7. Summarize Labels

In [ ]:
def summarize_llm_groundtruth_labels(llm_labels_path: Path) -> pd.DataFrame:
    """Summarize LLM-generated relevance labels before human validation."""
    llm_records = load_jsonl_records(llm_labels_path)
    label_dataframe = pd.DataFrame(llm_records)
    if label_dataframe.empty:
        raise ValueError(f"No labels found in {llm_labels_path}")

    print("Number of judged pairs:", len(label_dataframe))
    print("Number of queries:", label_dataframe["query_id"].nunique())
    print("Number of documents:", label_dataframe["doc_id"].nunique())
    print("\nOverall relevance distribution:")
    print(label_dataframe["relevance"].value_counts().sort_index())
    print("\nJudged pairs per query:")
    print(label_dataframe.groupby("query_id").size().describe())
    return label_dataframe


# Run after labeling completes.
# llm_groundtruth_dataframe = summarize_llm_groundtruth_labels(LLM_LABELS_PATH)